In [2]:
# ============================================
# LSTM SENTIMENT ANALYSIS USING PYTORCH
# ============================================

# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from collections import Counter


# ============================================
# 2. CHECK PYTORCH
# ============================================

print("Starting...", flush=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device, flush=True)
print("PyTorch Version:", torch.__version__, flush=True)


# ============================================
# 3. LOAD DATASET
# ============================================

df = pd.read_csv("IMDB Dataset.csv")

print("Dataset Shape:", df.shape, flush=True)
print("Columns:", df.columns.tolist(), flush=True)


# ============================================
# 4. CLEAN DATA
# ============================================

df = df.dropna()

df["review"] = df["review"].astype(str)

df["sentiment"] = (
    df["sentiment"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Convert sentiment into numerical labels
# Positive = 1
# Negative = 0

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

df = df.dropna(subset=["label"])

df["label"] = df["label"].astype(int)

print(
    "Positive Reviews:",
    int((df["label"] == 1).sum()),
    flush=True
)

print(
    "Negative Reviews:",
    int((df["label"] == 0).sum()),
    flush=True
)


# ============================================
# 5. TEXT CLEANING
# ============================================

def clean_text(text):

    text = text.lower()

    # Remove HTML tags
    text = re.sub(
        r"<br\s*/?>",
        " ",
        text
    )

    # Keep only alphabets and spaces
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


print("Cleaning reviews...", flush=True)

df["clean_review"] = df["review"].apply(
    clean_text
)

print("Cleaning completed.", flush=True)


# ============================================
# 6. TRAIN / VALIDATION / TEST SPLIT
# ============================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print(
    "Training Samples:",
    len(train_df),
    flush=True
)

print(
    "Validation Samples:",
    len(validation_df),
    flush=True
)

print(
    "Testing Samples:",
    len(test_df),
    flush=True
)


# ============================================
# 7. CREATE VOCABULARY
# ============================================

print("Creating vocabulary...", flush=True)

counter = Counter()

for text in train_df["clean_review"]:

    counter.update(
        text.split()
    )


max_vocab_size = 8000

word_to_index = {
    "<PAD>": 0,
    "<UNK>": 1
}

for index, (word, count) in enumerate(
    counter.most_common(max_vocab_size - 2),
    start=2
):

    word_to_index[word] = index


vocab_size = len(word_to_index)

print(
    "Vocabulary Size:",
    vocab_size,
    flush=True
)


# ============================================
# 8. ENCODE TEXT
# ============================================

max_length = 60


def encode_text(text):

    words = text.split()

    sequence = []

    for word in words[:max_length]:

        sequence.append(
            word_to_index.get(
                word,
                1
            )
        )

    while len(sequence) < max_length:

        sequence.append(0)

    return sequence


# ============================================
# 9. ENCODE DATASETS
# ============================================

print(
    "Encoding training reviews...",
    flush=True
)

train_sequences = []

for i, text in enumerate(
    train_df["clean_review"]
):

    train_sequences.append(
        encode_text(text)
    )

    if i % 5000 == 0:

        print(
            "Training reviews encoded:",
            i,
            flush=True
        )


print(
    "Encoding validation reviews...",
    flush=True
)

validation_sequences = [
    encode_text(text)
    for text in validation_df["clean_review"]
]


print(
    "Encoding test reviews...",
    flush=True
)

test_sequences = [
    encode_text(text)
    for text in test_df["clean_review"]
]

print(
    "Encoding completed.",
    flush=True
)


# ============================================
# 10. CREATE PYTORCH DATASET
# ============================================

class ReviewDataset(Dataset):

    def __init__(
        self,
        sequences,
        labels
    ):

        self.x = torch.tensor(
            sequences,
            dtype=torch.long
        )

        self.y = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(
        self,
        index
    ):

        return (
            self.x[index],
            self.y[index]
        )


# ============================================
# 11. CREATE DATASETS
# ============================================

train_dataset = ReviewDataset(
    train_sequences,
    train_df["label"].tolist()
)

validation_dataset = ReviewDataset(
    validation_sequences,
    validation_df["label"].tolist()
)

test_dataset = ReviewDataset(
    test_sequences,
    test_df["label"].tolist()
)


# ============================================
# 12. CREATE DATA LOADERS
# ============================================

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ============================================
# 13. DEFINE LSTM MODEL
# ============================================

class LSTMSentimentModel(
    nn.Module
):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            32,
            padding_idx=0
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=32,
            hidden_size=32,
            num_layers=1,
            batch_first=True
        )

        # Fully connected layer
        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, (
            hidden,
            cell
        ) = self.lstm(
            embedded
        )

        hidden = hidden[-1]

        output = self.fc(
            hidden
        )

        return output.squeeze(1)


# ============================================
# 14. CREATE MODEL
# ============================================

print(
    "Creating LSTM model...",
    flush=True
)

model = LSTMSentimentModel(
    vocab_size
)

model = model.to(device)

print(
    "Model created successfully.",
    flush=True
)

print(model)


# ============================================
# 15. LOSS FUNCTION AND OPTIMIZER
# ============================================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ============================================
# 16. TRAINING SETTINGS
# ============================================

epochs = 2

training_losses = []

validation_losses = []

best_validation_loss = float(
    "inf"
)


# ============================================
# 17. TRAIN MODEL
# ============================================

print(
    "\n=============================="
)

print(
    "TRAINING STARTED"
)

print(
    "==============================",
    flush=True
)


for epoch in range(epochs):

    model.train()

    total_training_loss = 0

    print(
        "\nStarting Epoch",
        epoch + 1,
        flush=True
    )


    for batch_number, (
        reviews,
        labels
    ) in enumerate(train_loader):

        reviews = reviews.to(device)

        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            reviews
        )

        # Calculate loss
        loss = criterion(
            outputs,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_training_loss += (
            loss.item()
        )

        if batch_number % 10 == 0:

            print(
                "Epoch",
                epoch + 1,
                "| Batch",
                batch_number + 1,
                "/",
                len(train_loader),
                flush=True
            )


    # Average training loss

    training_loss = (
        total_training_loss /
        len(train_loader)
    )


    # ========================================
    # VALIDATION
    # ========================================

    model.eval()

    total_validation_loss = 0


    with torch.no_grad():

        for reviews, labels in validation_loader:

            reviews = reviews.to(device)

            labels = labels.to(device)

            outputs = model(
                reviews
            )

            loss = criterion(
                outputs,
                labels
            )

            total_validation_loss += (
                loss.item()
            )


    validation_loss = (
        total_validation_loss /
        len(validation_loader)
    )


    training_losses.append(
        training_loss
    )

    validation_losses.append(
        validation_loss
    )


    print(
        "\nEpoch",
        epoch + 1,
        "completed",
        flush=True
    )

    print(
        "Training Loss:",
        round(training_loss, 4),
        flush=True
    )

    print(
        "Validation Loss:",
        round(validation_loss, 4),
        flush=True
    )


    # Save best model

    if validation_loss < best_validation_loss:

        best_validation_loss = (
            validation_loss
        )

        torch.save(
            model.state_dict(),
            "lstm_sentiment_model.pth"
        )


print(
    "\nTraining completed.",
    flush=True
)


# ============================================
# 18. LOAD BEST MODEL
# ============================================

model.load_state_dict(
    torch.load(
        "lstm_sentiment_model.pth",
        map_location=device
    )
)

model.eval()


# ============================================
# 19. MODEL EVALUATION
# ============================================

print(
    "\nEvaluating model...",
    flush=True
)

predictions = []

actual = []


with torch.no_grad():

    for reviews, labels in test_loader:

        reviews = reviews.to(device)

        outputs = model(
            reviews
        )

        probabilities = torch.sigmoid(
            outputs
        )

        predicted = (
            probabilities >= 0.5
        ).int()

        predictions.extend(
            predicted.cpu().numpy()
        )

        actual.extend(
            labels.numpy()
        )


# ============================================
# 20. CALCULATE METRICS
# ============================================

accuracy = accuracy_score(
    actual,
    predictions
)

precision = precision_score(
    actual,
    predictions,
    zero_division=0
)

recall = recall_score(
    actual,
    predictions,
    zero_division=0
)

f1 = f1_score(
    actual,
    predictions,
    zero_division=0
)


# ============================================
# 21. DISPLAY MODEL PERFORMANCE
# ============================================

print(
    "\n=============================="
)

print(
    "MODEL PERFORMANCE"
)

print(
    "=============================="
)

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-Score  : {f1 * 100:.2f}%"
)


# ============================================
# 22. DISPLAY TRAINING / VALIDATION LOSS
# ============================================

print(
    "\n=============================="
)

print(
    "TRAINING / VALIDATION LOSS"
)

print(
    "=============================="
)


for i in range(epochs):

    print(
        f"Epoch {i + 1}: "
        f"Training Loss = "
        f"{training_losses[i]:.4f}, "
        f"Validation Loss = "
        f"{validation_losses[i]:.4f}"
    )


# ============================================
# 23. PREDICT SENTIMENT OF NEW REVIEW
# ============================================

def predict_sentiment(
    review
):

    model.eval()

    cleaned_review = clean_text(
        review
    )

    sequence = encode_text(
        cleaned_review
    )

    sequence = torch.tensor(
        [sequence],
        dtype=torch.long
    )

    sequence = sequence.to(device)


    with torch.no_grad():

        output = model(
            sequence
        )

        probability = torch.sigmoid(
            output
        ).item()


    if probability >= 0.5:

        sentiment = "Positive"

        confidence = (
            probability * 100
        )

    else:

        sentiment = "Negative"

        confidence = (
            (1 - probability) * 100
        )


    print(
        "\n=============================="
    )

    print(
        "NEW REVIEW PREDICTION"
    )

    print(
        "=============================="
    )

    print(
        "Input:",
        review
    )

    print(
        "Predicted Sentiment:",
        sentiment
    )

    print(
        f"Confidence: {confidence:.2f}%"
    )


# ============================================
# 24. TEST WITH NEW REVIEW
# ============================================

predict_sentiment(
    "The movie was excellent and very enjoyable"
)

Starting...
Device: cpu
PyTorch Version: 2.12.1+cpu


FileNotFoundError: [Errno 2] No such file or directory: 'IMDB Dataset.csv'